In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
from scipy.stats import gaussian_kde
from tqdm import tqdm
import warnings
from skimage import util
from skimage.filters import threshold_otsu, gaussian, threshold_isodata, threshold_minimum, threshold_triangle, threshold_otsu, threshold_mean, threshold_li, try_all_threshold, threshold_yen, threshold_multiotsu
from skimage.segmentation import clear_border, watershed, expand_labels
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, opening, ball, dilation, erosion
from skimage.transform import rescale
from skimage.feature import peak_local_max
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi
from scipy import stats as _stats
from matplotlib.colors import to_hex
from skimage.filters import median as median_filter
from skimage.morphology import square
from tifffile import imread
import tifffile
import napari
import colorsys
from collections import defaultdict
from skimage.morphology import closing, disk
import contextlib
from skimage.exposure import rescale_intensity

from PIL import ImageColor
from vtkmodules.vtkFiltersGeneral import vtkDiscreteFlyingEdges3D, vtkDiscreteMarchingCubes

import vtk
from vtk.util.numpy_support import numpy_to_vtk, vtk_to_numpy


In [18]:
# segmented = tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom3\segmented.tif")
# image_fluo= tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom3\Vimenting_SMA_VE-cadherin.lif - Ve-cad A488_SMA A594_zoom_3.tif")

segmented = tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom4\just_vasculature.tif")
image_fluo= tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom4\all_staining.tif")


fibroblasts = image_fluo[:,1,:,:]

binary = segmented > 0

# Gaussian smooth then re-threshold to smooth edges
smoothed = gaussian(binary.astype(float), sigma=1)
binary_smooth = smoothed > 0.5

# Slice-by-slice: remove small objects (area < 100) in each 2D slice
cleaned = np.zeros_like(binary_smooth, dtype=bool)
for z in range(binary_smooth.shape[0]):
    sl = binary_smooth[z]
    sl_labelled = label(sl)
    sl_cleaned = remove_small_objects(sl_labelled, min_size=1000)
    cleaned[z] = sl_cleaned > 0

# Relabel in 3D and keep only the largest component
labelled = label(cleaned)
props = pd.DataFrame(regionprops_table(labelled, properties=("label", "area")))
largest_label = int(props.loc[props["area"].idxmax(), "label"])
labelled = (labelled == largest_label).astype(np.int32) * largest_label
print(f"Kept label {largest_label}, area = {props['area'].max()} voxels")

smoothed = gaussian(labelled, sigma=3, preserve_range=True)
binary_smoothed = smoothed > threshold_otsu(smoothed)



C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_600372\2301640185.py:21: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  sl_cleaned = remove_small_objects(sl_labelled, min_size=1000)


Kept label 1, area = 4675842.0 voxels


In [19]:
viewer = napari.Viewer()
# labels_layer = viewer.add_labels(binary_smoothed.astype(np.int32), scale=(5,2.2750,2.2750))
labels_layer = viewer.add_labels(binary_smoothed.astype(np.uint32), scale = (5,1.13,1.13))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'red'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'magenta'}
# viewer.add_image(fibroblasts, scale=(5,2.2750,2.2750))
viewer.add_image(fibroblasts, scale=(5, 1.13,1.13))

<Image layer 'fibroblasts' at 0x21d956c0450>

In [30]:
segmented = tifffile.imread(r"Z:\Marina\Stellaris\2026.04.01_CART_setup\Bel\TEST\segmentation.tif")
image_fluo= tifffile.imread(r"Z:\Marina\Stellaris\2026.04.01_CART_setup\Bel\TEST\all_channels.tif")

# Downsample by 4 in all spatial dims
DS = 4
segmented = segmented[::DS, ::DS, ::DS]
image_fluo = image_fluo[::DS, :, ::DS, ::DS]

cart = image_fluo[:,0,:,:]

binary = segmented > 0

# Gaussian smooth then re-threshold to smooth edges
smoothed = gaussian(binary.astype(float), sigma=1)
binary_smooth = smoothed > 0.5

# Slice-by-slice: remove small objects (area < 100) in each 2D slice
cleaned = np.zeros_like(binary_smooth, dtype=bool)
for z in range(binary_smooth.shape[0]):
    sl = binary_smooth[z]
    sl_labelled = label(sl)
    sl_cleaned = remove_small_objects(sl_labelled, min_size=1000 // (DS * DS))
    cleaned[z] = sl_cleaned > 0

# Relabel in 3D and keep only the largest component
labelled = label(cleaned)
props = pd.DataFrame(regionprops_table(labelled, properties=("label", "area")))
largest_label = int(props.loc[props["area"].idxmax(), "label"])
labelled = (labelled == largest_label).astype(np.int32) * largest_label
print(f"Kept label {largest_label}, area = {props['area'].max()} voxels")

smoothed = gaussian(labelled, sigma=3, preserve_range=True)
binary_smoothed = smoothed > threshold_otsu(smoothed)
print(f"Downsampled shape: {cart.shape}")

C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_600372\1737231175.py:22: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  sl_cleaned = remove_small_objects(sl_labelled, min_size=1000 // (DS * DS))


Kept label 1, area = 915543.0 voxels
Downsampled shape: (28, 719, 718)


In [37]:
# Add 20 slices of black padding on each side in Z
Z_PAD = 20
pad_seg = np.zeros((Z_PAD, *binary_smoothed.shape[1:]), dtype=binary_smoothed.dtype)
binary_smoothed = np.concatenate([pad_seg, binary_smoothed, pad_seg], axis=0)

pad_cart = np.zeros((Z_PAD, *cart.shape[1:]), dtype=cart.dtype)
cart = np.concatenate([pad_cart, cart, pad_cart], axis=0)

print(f"Padded shapes: cart={cart.shape}, segmentation={binary_smoothed.shape}")


Padded shapes: cart=(68, 719, 718), segmentation=(68, 719, 718)


In [38]:
# --- Ellipsoid at intensity-weighted COM of CART channel ---
# Adjust these radii (in voxels) by trial and error
radius_z = 30    # half-length along Z
radius_y = 90    # half-length along Y
radius_x = 90    # half-length along X

# Compute intensity-weighted centre of mass
com = ndi.center_of_mass(cart)
cz, cy, cx = com
cz -= 2
cx += 10
cy += 10
print(f"CART intensity COM: z={cz:.1f}, y={cy:.1f}, x={cx:.1f}")

# Build coordinate grids and evaluate ellipsoid equation
zz, yy, xx = np.ogrid[:cart.shape[0], :cart.shape[1], :cart.shape[2]]
ellipsoid_mask = (((zz - cz) / radius_z) ** 2 +
                  ((yy - cy) / radius_y) ** 2 +
                  ((xx - cx) / radius_x) ** 2) <= 1.0

cart_ellipsoid = ellipsoid_mask.astype(np.int32)
print(f"Ellipsoid voxels: {cart_ellipsoid.sum()}")


CART intensity COM: z=29.8, y=363.9, x=364.4
Ellipsoid voxels: 1017861


In [39]:
viewer = napari.Viewer()
labels_layer = viewer.add_labels(cart_ellipsoid.astype(np.int32), scale=(2,1.1018,1.1018))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'gray'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'magenta'}
labels_layer = viewer.add_labels(binary_smoothed.astype(np.int32), scale=(2,1.1018,1.1018))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'red'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'darkorange'}
viewer.add_image(cart, scale=(2,1.1018,1.1018) )

<Image layer 'cart' at 0x21db949c750>

In [2]:
def tif_pixel_sizes(tif):
    """Return (z_um, y_um, x_um) from a tifffile TiffFile; any value is None if unavailable."""
    z_um = y_um = x_um = None
    tags = {t.name: t.value for t in tif.pages[0].tags.values()}
    xr = tags.get("XResolution")
    yr = tags.get("YResolution")
    if xr and xr[0]:
        x_um = xr[1] / xr[0]  # XResolution = pixels per unit -> um per pixel
    if yr and yr[0]:
        y_um = yr[1] / yr[0]
    ij = tif.imagej_metadata or {}
    if "spacing" in ij:
        z_um = float(ij["spacing"])
    return z_um, y_um, x_um

In [11]:
image_path = r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_REPRESENTATIVE_IMAGE_OUTPUTS\Bel_mCh_BF_VASCUMAP_FL32_UTD4_Merged_6channel.tif"


In [12]:
test = imread(pathlib.Path(image_path))
with tifffile.TiffFile(image_path) as tif:
    z_um, y_um, x_um = tif_pixel_sizes(tif)
    print(f"Pixel sizes (z, y, x): {z_um}, {y_um}, {x_um}")
seg = test[:,-2,:,:].astype(np.uint8)
fluo = test[:,1,:,:]
bf = test[:,0,:,:]
# ------------------------------------------------------------
# NumPy -> vtkImageData
# ------------------------------------------------------------

z, y, x = seg.shape

vtk_image = vtk.vtkImageData()
vtk_image.SetDimensions(x, y, z)

# VTK coordinates are X, Y, Z
# Your napari scale is Z, Y, X = (10, 2, 2)
vtk_image.SetSpacing(x_um, y_um, z_um)

vtk_array = numpy_to_vtk(seg.ravel(order="C"), deep=True, array_type=vtk.VTK_UNSIGNED_CHAR,)

vtk_image.GetPointData().SetScalars(vtk_array)


surface_nets = vtk.vtkSurfaceNets3D()
surface_nets.SetInputData(vtk_image)

# Extract label 1
surface_nets.SetValue(0, 1)

# Built-in smoothing
surface_nets.SetSmoothing(True)
surface_nets.SetNumberOfIterations(500)

# Napari needs triangles
surface_nets.SetOutputMeshTypeToTriangles()

surface_nets.Update()

mesh = surface_nets.GetOutput()

# Decimate the mesh so napari renders it quickly (removes ~97% of triangles)
print(f"Triangles before decimation: {mesh.GetNumberOfPolys()}")
decimate = vtk.vtkQuadricDecimation()
decimate.SetInputData(mesh)
decimate.SetTargetReduction(0.97)
decimate.Update()
mesh = decimate.GetOutput()
print(f"Triangles after decimation: {mesh.GetNumberOfPolys()}")

# VTK vertices are X, Y, Z
vertices_xyz = vtk_to_numpy(mesh.GetPoints().GetData())

# Napari volume coordinates are Z, Y, X
vertices = vertices_xyz[:, [2, 1, 0]]

# vtkCellArray format:
# [3, i0, i1, i2,
#  3, i0, i1, i2,
#  ...]
polys = vtk_to_numpy(mesh.GetPolys().GetData())
faces = polys.reshape(-1, 4)[:, 1:]



Pixel sizes (z, y, x): 9.99954, 1.3000000000605358, 1.3000000000605358
Triangles before decimation: 38179038
Triangles after decimation: 38174487


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_72044\2706240130.py:62: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  polys = vtk_to_numpy(mesh.GetPolys().GetData())


In [13]:
fluo = rescale_intensity(fluo, out_range=(0, 255)).astype(np.uint8)
p_low, p_high = np.percentile(bf, (1, 99))
bf = rescale_intensity(bf, in_range=(p_low, p_high), out_range=(0, 255)).astype(np.uint8)

In [16]:
# Render from XY-downsampled volumes; scale is multiplied by DS to keep the
# physical size, so they still line up with the surface (already in microns).
DS = 2
# Copy to contiguous arrays so napari doesn't re-copy strided views on every refresh
fluo_r = np.ascontiguousarray(fluo[:, ::DS, ::DS])
bf_r = np.ascontiguousarray(bf[:, ::DS, ::DS])
seg_r = np.ascontiguousarray(seg[:, ::DS, ::DS])
render_scale = (z_um, y_um * DS, x_um * DS)




In [ ]:
viewer = napari.Viewer()
viewer.window._qt_viewer.canvas.native.setFixedSize(1280, 720)
viewer.add_surface((vertices, faces), name="smoothed segmentation", shading="smooth",)
# viewer.add_image(fluo_r, scale=render_scale, name="fluorescence")
viewer.add_image(bf_r, scale=render_scale, name="brightfield")
# 3D labels rendering is slow and duplicates the surface; hidden by default
viewer.add_labels(seg_r, scale=render_scale, name="segmentation", visible=False)

<Labels layer 'segmentation' at 0x27874361c60>

Rendering frames...


100%|██████████| 131/131 [00:25<00:00,  5.15it/s]


Rendering frames...


100%|██████████| 241/241 [00:29<00:00,  8.27it/s]


Rendering frames...


 42%|████▏     | 151/361 [00:25<00:35,  5.88it/s]


In [6]:
def create_triangular_mesh(seg, x_um, y_um, z_um):
    z, y, x = seg.shape

    vtk_image = vtk.vtkImageData()
    vtk_image.SetDimensions(x, y, z)

    # VTK coordinates are X, Y, Z
    # Your napari scale is Z, Y, X = (10, 2, 2)
    vtk_image.SetSpacing(x_um, y_um, z_um)

    vtk_array = numpy_to_vtk(seg.ravel(order="C"), deep=True, array_type=vtk.VTK_UNSIGNED_CHAR,)

    vtk_image.GetPointData().SetScalars(vtk_array)


    surface_nets = vtk.vtkSurfaceNets3D()
    surface_nets.SetInputData(vtk_image)

    # Extract label 1
    surface_nets.SetValue(0, 1)

    # Built-in smoothing
    surface_nets.SetSmoothing(True)
    surface_nets.SetNumberOfIterations(500)

    # Napari needs triangles
    surface_nets.SetOutputMeshTypeToTriangles()

    surface_nets.Update()

    mesh = surface_nets.GetOutput()

    # Decimate the mesh so napari renders it quickly (removes ~97% of triangles)
    print(f"Triangles before decimation: {mesh.GetNumberOfPolys()}")
    decimate = vtk.vtkQuadricDecimation()
    decimate.SetInputData(mesh)
    decimate.SetTargetReduction(0.97)
    decimate.Update()
    mesh = decimate.GetOutput()
    print(f"Triangles after decimation: {mesh.GetNumberOfPolys()}")

    # VTK vertices are X, Y, Z
    vertices_xyz = vtk_to_numpy(mesh.GetPoints().GetData())

    # Napari volume coordinates are Z, Y, X
    vertices = vertices_xyz[:, [2, 1, 0]]

    # vtkCellArray format:
    # [3, i0, i1, i2,
    #  3, i0, i1, i2,
    #  ...]
    polys = vtk_to_numpy(mesh.GetPolys().GetData())
    faces = polys.reshape(-1, 4)[:, 1:]
    return vertices, faces


In [4]:
untreated_path_v2 = pathlib.Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260615_Exp7_Vascumap_BF_RFP_Thunder_DMSO_M1-7_DMSO_6channel.tif")
untreated_path_v1 = pathlib.Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V1_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260615_Exp7_Vascumap_BF_RFP_Thunder_DMSO_M1-7_DMSO_6channel.tif")
pax_path_v2 = pathlib.Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Paclitaxel_M2-3_Merged_6channel.tif")
pax_path_v1 = pathlib.Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_~RETRAINING\V1_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Paclitaxel_M2-3_Merged_6channel.tif")
ibuprofen_path_v1 = pathlib.Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V1_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Ibuprofen_M1-8_Merged_6channel.tif")
ibuprofen_path_v2 = pathlib.Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Ibuprofen_M1-8_Merged_6channel.tif")
with tifffile.TiffFile(untreated_path_v2) as tif:
    z_um, y_um, x_um = tif_pixel_sizes(tif)
    print(f"Pixel sizes (z, y, x): {z_um}, {y_um}, {x_um}")



Pixel sizes (z, y, x): 9.99954, 1.3000000000605358, 1.3000000000605358


In [5]:
v1_path = untreated_path_v1
v2_path = untreated_path_v2
v1 = tifffile.imread(v1_path)
v2 = tifffile.imread(v2_path)
v1_seg = v1[:,-2,:,:].astype(np.uint8)
v2_seg = v2[:,-2,:,:].astype(np.uint8)
fluo = v1[:,1,:,:].astype(np.uint8)
bf = v1[:,0,:,:].astype(np.uint8)

p_low, p_high = np.percentile(bf, (1, 99))
bf = rescale_intensity(bf, in_range=(p_low, p_high), out_range=(0, 255)).astype(np.uint8)
p_low, p_high = np.percentile(fluo, (1, 99))
fluo = rescale_intensity(fluo, in_range=(p_low, p_high), out_range=(0, 255)).astype(np.uint8)
v1 = 0
v2 = 0

In [7]:
v1_vertices, v1_faces = create_triangular_mesh(v1_seg, x_um, y_um, z_um)
v2_vertices, v2_faces = create_triangular_mesh(v2_seg, x_um, y_um, z_um)


Triangles before decimation: 59208366
Triangles after decimation: 59202610


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_7412\4244810380.py:52: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  polys = vtk_to_numpy(mesh.GetPolys().GetData())


Triangles before decimation: 24589870
Triangles after decimation: 737696


In [ ]:
viewer = napari.Viewer()
viewer.window._qt_viewer.canvas.native.setFixedSize(1280, 720)
# viewer.add_image(fluo_r, scale=render_scale, name="fluorescence")
viewer.add_image(bf, scale=(z_um, y_um, x_um), name="brightfield")
# 3D labels rendering is slow and duplicates the surface; hidden by default
# viewer.add_labels(seg_r, scale=render_scale, name="segmentation", visible=False)
viewer.add_surface((v1_vertices, v1_faces), name="v1 segmentation", shading="smooth",)
viewer.add_surface((v2_vertices, v2_faces), name="v2 segmentation", shading="smooth",)

<Surface layer 'v2 segmentation' at 0x2b97571a9b0>

Rendering frames...


100%|██████████| 136/136 [00:58<00:00,  2.33it/s]
